# 🏛️ TourBot — Alexandria Multi-Agent Tourism Chatbot

**Stack:** Groq (Llama free models) + BGE-M3 local embeddings + ChromaDB + LangChain Tool-Calling Agent + FastAPI

**Agents (all tools on one orchestrator):**
1. 📚 `search_tourism_knowledge` — places, history, attractions (RAG)
2. 🌤️ `get_weather` — live weather, no API key (Open-Meteo)
3. 🍽️ `food_experience_advisor` — restaurant recs, asks cuisine + budget first
4. 🏨 `hotel_advisor` — hotel/accommodation recs, grounded only in retrieved data
5. ⚖️ `compare_places` — side-by-side Markdown comparison table
6. 🎯 `apply_travel_constraints` — adapts for kids/solo/budget/couples/elderly

**Data sources:** official PDFs + official hotel/restaurant/tourism websites for Alexandria only — no random blogs, to minimize hallucination risk.

**Embeddings:** built locally with `BAAI/bge-m3` — **run the embedding step ONCE**, then the vectorstore persists to disk in `tourbot_vectordb/`. Every future session just loads it back instantly — no re-embedding, no re-running the model.

### Keys needed (add to Colab Secrets 🔑):
- `GROQ_API_KEY` → free at **console.groq.com**
- `NGROK_TOKEN` → free at **dashboard.ngrok.com/get-started/your-authtoken**

> No OpenRouter/Google key needed — embeddings run locally and free, LLM is Groq's free tier.

---
## 1. Install Dependencies

In [1]:
!pip install -q "langchain==0.2.16" "langchain-core==0.2.43" "langchain-groq==0.1.9" "langchain-community==0.2.16" langchain-text-splitters
!pip install -q chromadb pypdf docx2txt fastapi uvicorn nest-asyncio pyngrok
!pip install -q trafilatura geopy sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 50.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38

> ⚠️ **Restart the session now** (Runtime → Restart Session), then run every cell below in order.

---
## 2. Imports, API Keys, LLM & Embeddings

- **LLM:** `llama-3.1-8b-instant` on Groq — free, fast, separate (higher) daily token quota than the 70B model, good quality for tool-calling + tourism Q&A.
- **Embeddings:** `BAAI/bge-m3` running locally via `sentence-transformers` — completely free, no API key, no rate limits, strong multilingual quality. Built **once**, then reused from disk forever after.

In [1]:
import os
import re
import glob
import time
import shutil
import textwrap
import threading
import requests
from typing import List
from google.colab import userdata

# --- API Keys ---
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["NGROK_TOKEN"]  = userdata.get("NGROK_TOKEN")

# --- LangChain core ---
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings

# --- Agent ---
from langchain.agents import create_tool_calling_agent, AgentExecutor

# --- Vectorstore & loaders ---
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader, TextLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- LLM ---
from langchain_groq import ChatGroq

# --- Local embeddings (free, no rate limits) ---
from sentence_transformers import SentenceTransformer

# --- Other tools ---
from geopy.geocoders import Nominatim
import trafilatura

print("✅ Imports done!")

# ----------------------------------------------------------------
# LLM — free + fast Groq model
# ----------------------------------------------------------------
MODEL_NAME = "openai/gpt-oss-120b"

llm = ChatGroq(
    model=MODEL_NAME,
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.3
)

# ----------------------------------------------------------------
# Embeddings — local, free, no API key
# ----------------------------------------------------------------
class BGEEmbeddings(Embeddings):
    def __init__(self):
        self.model = SentenceTransformer("BAAI/bge-m3")

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode(texts, show_progress_bar=True, batch_size=16).tolist()

    def embed_query(self, text: str) -> List[float]:
        return self.model.encode([text])[0].tolist()

embedding = BGEEmbeddings()

print(f"✅ LLM ready        → {MODEL_NAME} on Groq (free)")
print(f"✅ Embeddings ready  → BAAI/bge-m3 (local, free)")

✅ Imports done!


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ LLM ready        → openai/gpt-oss-120b on Groq (free)
✅ Embeddings ready  → BAAI/bge-m3 (local, free)


---
## 3. Data Sources — Alexandria Only

Official, named sources to minimize hallucination — no generic blogs as the primary source, no social media (Facebook/TripAdvisor don't scrape reliably and were dropped):
- **PDFs**: drop any official Alexandria tourism/hotel/restaurant PDFs into `tourism_docs/`
- **Websites**: official monument/museum pages, hotel dining pages, Wikipedia, official tourism guides

In [2]:
!mkdir -p tourism_docs

In [4]:
# ----------------------------------------------------------------
# Loader functions — PDFs (folder) + websites
# ----------------------------------------------------------------
def load_folder(folder_path):
    """Load every PDF/TXT/DOCX inside a folder (recursive)."""
    docs = []
    patterns = ["**/*.pdf", "**/*.txt", "**/*.docx"]
    files = []
    for p in patterns:
        files.extend(glob.glob(os.path.join(folder_path, p), recursive=True))

    print(f"📂 Found {len(files)} files in {folder_path}")
    for path in files:
        try:
            if path.endswith(".pdf"):
                docs.extend(PyPDFLoader(path).load())
            elif path.endswith(".txt"):
                docs.extend(TextLoader(path).load())
            elif path.endswith(".docx"):
                docs.extend(Docx2txtLoader(path).load())
            print(f"  ✅ {os.path.basename(path)}")
        except Exception as e:
            print(f"  ❌ {os.path.basename(path)} — {e}")
    return docs

def load_websites(urls):
    docs = []
    for url in urls:
        try:
            downloaded = trafilatura.fetch_url(url)
            text = trafilatura.extract(downloaded)
            if text:
                docs.append(Document(page_content=text, metadata={"source": url, "type": "website"}))
                print(f"  ✅ {url} ({len(text)} chars)")
            else:
                print(f"  ⚠️  No content extracted: {url}")
        except Exception as e:
            print(f"  ❌ {url} — {e}")
    return docs

print("✅ Loader functions ready")

✅ Loader functions ready


In [5]:
# ----------------------------------------------------------------
# EDIT THESE — your real Alexandria sources
# ----------------------------------------------------------------
PDF_FOLDER = "/content/tourism_docs"   # drop any official PDFs here (governorate guides, hotel brochures, etc)

# General attractions / history / official tourism
WEBSITE_URLS_PLACES = [
    "https://www.egypttoursplus.com/visiting-alexandria-egypt/",
    "https://www.bibalex.org/en/default",
    "https://egymonuments.gov.eg/en/archaeological-sites/alexandria",
    "https://egymonuments.gov.eg/en/monuments/the-cemetery-of-shatby",
    "https://egymonuments.gov.eg/en/monuments/qaitbay-fort",
    "https://egymonuments.gov.eg/en/museums/alexandria-national-museum",
    "https://egymonuments.gov.eg/en/collections/alexandria",
    "https://egymonuments.gov.eg/en/museums/graeco-roman-museum",
    "https://egymonuments.gov.eg/en/news/eliyahu-hanavi-synagogue-in-alexandria-reopened",
    "https://egymonuments.gov.eg/en/news/opening-of-the-qaitbay-citadel-visitor-center-in-alexandria",
    "https://egymonuments.gov.eg/en/sunken-monuments/abu-qir-bay",
    "https://egymonuments.gov.eg/en/news/visit-of-the-thai-foreign-minister-to-greco-roman-museum-in-alexandria",
    "https://egymonuments.gov.eg/en/archaeological-sites/kom-el-dikka",
    "https://egymonuments.gov.eg/en/sunken-monuments/shatby",
    "https://egymonuments.gov.eg/en/sunken-monuments/mamoura-bay",
    "https://egymonuments.gov.eg/en/archaeological-sites/abu-mina-1",
    "https://egymonuments.gov.eg/en/discover-egyptian-treasure",
    "https://en.wikipedia.org/wiki/Montaza_Palace",
    "https://en.wikipedia.org/wiki/Citadel_of_Qaitbay",
    "https://en.wikipedia.org/wiki/Ras_El_Tin_Palace",
    "https://www.egypttoursgate.com/blog/egypt-travel-news/alexandria-aquarium-museum",
    "https://en.wikipedia.org/wiki/Mahatet_El_Raml",
    "https://egymonuments.gov.eg/en/museums/royal-jewelry-museum",
    "https://en.wikipedia.org/wiki/Catacombs_of_Kom_El_Shoqafa",
    "https://ar.wikipedia.org/wiki/%D8%B9%D9%85%D9%88%D8%AF_%D8%A7%D9%84%D8%B3%D9%88%D8%A7%D8%B1%D9%8A",
    "https://en.wikipedia.org/wiki/Alexandria_National_Museum",
    "https://www.cairoopera.org/en/theaters/alexandria-opera-house-sayed-darwish-theatre/",
    "https://en.wikipedia.org/wiki/San_Stefano_Grand_Plaza",
    "https://ar.wikipedia.org/wiki/%D8%AC%D8%B1%D9%8A%D9%86_%D8%A8%D9%84%D8%A7%D8%B2%D8%A7",
    "https://en.wikipedia.org/wiki/Alexandria_Corniche",
    "https://ar.wikipedia.org/wiki/%D9%85%D8%B3%D8%AC%D8%AF_%D8%A3%D8%A8%D9%88_%D8%A7%D9%84%D8%B9%D8%A8%D8%A7%D8%B3_%D8%A7%D9%84%D9%85%D8%B1%D8%B3%D9%8A",
    "https://ar.wikipedia.org/wiki/%D9%86%D8%B5%D8%A8_%D8%A7%D9%84%D8%AC%D9%86%D8%AF%D9%8A_%D8%A7%D9%84%D9%85%D8%AC%D9%87%D9%88%D9%84_(%D8%A7%D9%84%D8%A5%D8%B3%D9%83%D9%86%D8%AF%D8%B1%D9%8A%D8%A9)",
    "https://www.bibalex.org/alexmed/gastronomy/Eateries/Details.aspx?id=OWVuuxxm5GiAtDAfe39TOw%3D%3D",
    "https://www.roughguides.com/egypt/alexandria-mediterranean-coast-delta/alexandria/",
    "https://www.roughguides.com/egypt/alexandria-mediterranean-coast-delta/",
]

WEBSITE_URLS_RESTAURANTS = [
    "https://www.fourseasons.com/alexandria/dining/",
    "https://www.fourseasons.com/alexandria/dining/restaurants/beach_restaurant_and_lounge/",
    "https://www.fourseasons.com/alexandria/dining/seasonal-dining-experiences/",
    "https://jeedas.com/menu/",
    "https://wanderlog.com/list/geoCategory/75347/where-to-eat-best-restaurants-in-alexandria"
]

WEBSITE_URLS_HOTELS = [
    "https://www.fourseasons.com/alexandria/",
    "https://www.hilton.com/en/offers/",
    "https://www.hilton.com/en/events/",
    "https://paradiseinnegypt.com/windsor-palace-hotel/dining/",
    "https://www.sunrise-resorts.com/alex-avenue-hotel",
    "https://hrewards.com/en/steigenberger-cecil-hotel-alexandria",
    "https://www.helnan.com/en/",
    "https://www.ihg.com/crowneplaza/hotels/gb/en/alexandria/alysm/hoteldetail",
    "https://tolipgroup.com/tolip-hotel-royal-alexandria/",
]

WEBSITE_URLS = WEBSITE_URLS_PLACES + WEBSITE_URLS_RESTAURANTS + WEBSITE_URLS_HOTELS

print("📚 Loading PDFs...")
pdf_docs = load_folder(PDF_FOLDER)

print("\n🌐 Loading websites...")
website_docs = load_websites(WEBSITE_URLS)

all_docs = pdf_docs + website_docs
print(f"\n📦 Total documents loaded: {len(all_docs)}")
print(f"   - PDFs/files: {len(pdf_docs)}")
print(f"   - Websites:   {len(website_docs)}")

📚 Loading PDFs...
📂 Found 12 files in /content/tourism_docs
  ✅ part of the book.pdf
  ✅ Abukir (2).pdf
  ✅ regulations-for-the-resumption-of-toursim-to-egypt.pdf
  ✅ 10010093 (1).pdf
  ✅ Lido Menu.pdf
  ✅ 060-ptolemaic-egypt-a-travelers-guide-to-alexandria-transcript.pdf
  ✅ 206-Article Text-452-1-10-20190424.pdf
  ✅ Carlos-Menu-Food-Service-Add.pdf
  ✅ download.pdf
  ✅ 2023061512554711546_Annualreport6202172022web.pdf
  ✅ The_Touristic_Development_of_the_Archaeo.pdf
  ✅ Octa Menu.pdf

🌐 Loading websites...
  ✅ https://www.egypttoursplus.com/visiting-alexandria-egypt/ (13667 chars)
  ✅ https://www.bibalex.org/en/default (6187 chars)
  ✅ https://egymonuments.gov.eg/en/archaeological-sites/alexandria (2388 chars)
  ✅ https://egymonuments.gov.eg/en/monuments/the-cemetery-of-shatby (854 chars)
  ✅ https://egymonuments.gov.eg/en/monuments/qaitbay-fort (1166 chars)
  ✅ https://egymonuments.gov.eg/en/museums/alexandria-national-museum (4234 chars)
  ✅ https://egymonuments.gov.eg/en/collectio

---
## 4. Build (or Reload) the Vectorstore

**Run this ONCE.** It checks whether a saved vectorstore already exists in `tourbot_vectordb/`:
- If it exists → loads instantly, skips embedding entirely (fast, free, no recomputation)
- If it doesn't exist → embeds `all_docs` with BGE-M3 and saves to disk for every future run

To force a fresh re-embed later (e.g. after adding new sources), just delete the `tourbot_vectordb/` folder before running this cell again.

In [6]:
import shutil
shutil.unpack_archive("/content/tourbot_vectordb_backup.zip", "/content/tourbot_vectordb")

In [8]:
PERSIST_DIR = "./tourbot_vectordb"

def split_and_store(docs, persist_dir=PERSIST_DIR):
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    splits = splitter.split_documents(docs)
    print(f"📄 Total chunks to embed: {len(splits)}")
    vectordb = Chroma.from_documents(splits, embedding, persist_directory=persist_dir)
    print(f"💾 Vectorstore saved to {persist_dir} — reusable from now on, no need to re-embed")
    return vectordb

def load_existing_vectordb(persist_dir=PERSIST_DIR):
    return Chroma(persist_directory=persist_dir, embedding_function=embedding)

if os.path.exists(PERSIST_DIR) and os.listdir(PERSIST_DIR):
    print(f"📦 Found existing vectorstore at {PERSIST_DIR} — loading instantly (no re-embedding)")
    vectordb = load_existing_vectordb()
else:
    print("🆕 No existing vectorstore found — embedding now (one-time cost)")
    vectordb = split_and_store(all_docs)

print("\n✅ Knowledge base ready!")

📦 Found existing vectorstore at ./tourbot_vectordb — loading instantly (no re-embedding)


/tmp/ipykernel_8748/1181682617.py:12: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  return Chroma(persist_directory=persist_dir, embedding_function=embedding)



✅ Knowledge base ready!


In [9]:
import shutil
from google.colab import files

# Zip the entire vectorstore folder
shutil.make_archive("tourbot_vectordb_backup", "zip", "tourbot_vectordb")

# Trigger browser download
files.download("tourbot_vectordb_backup.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 5. Agent Tools

All 6 tools (RAG search, weather, food advisor, hotel advisor, place comparator, travel constraints) defined together. The single orchestrator agent below decides which to call per message — this is the standard "one agent, multiple tools" pattern, equivalent in practice to having separate specialist agents.

In [10]:
geolocator = Nominatim(user_agent="tourbot_egypt")

# ==================================================================
# TOOL 1 — RAG Knowledge Search
# ==================================================================
@tool
def search_tourism_knowledge(query: str) -> str:
    """Search the Alexandria tourism knowledge base (official PDFs and official tourism/hotel
    websites) for information about tourist places, attractions, history, museums, activities,
    and travel tips. Use this whenever the user asks about a specific place, activity, or
    general tourism information about Alexandria."""
    retriever = vectordb.as_retriever(search_kwargs={"k": 4})
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant information found in the knowledge base."
    return "\n\n".join([d.page_content for d in docs])


# ==================================================================
# TOOL 2 — Live Weather (Open-Meteo, free, no API key)
# ==================================================================
@tool
def get_weather(city: str) -> str:
    """Get the current real-time weather for a city in Egypt (e.g. Alexandria, Cairo, Luxor).
    Returns temperature, humidity, wind speed, and conditions. Use this whenever the user asks
    about weather, what to wear, or wants activity recommendations that depend on conditions."""
    try:
        location = geolocator.geocode(f"{city}, Egypt")
        if not location:
            return f"Could not find location: {city}"

        lat, lon = location.latitude, location.longitude
        url = (
            f"https://api.open-meteo.com/v1/forecast"
            f"?latitude={lat}&longitude={lon}"
            f"&current=temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code"
            f"&timezone=auto"
        )
        response = requests.get(url, timeout=10)
        current = response.json()["current"]

        weather_codes = {
            0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
            45: "Fog", 51: "Light drizzle", 61: "Light rain", 63: "Moderate rain",
            80: "Rain showers", 95: "Thunderstorm"
        }
        condition = weather_codes.get(current["weather_code"], "Unknown")

        return (
            f"Weather in {city}, Egypt right now:\n"
            f"- Temperature: {current['temperature_2m']}°C\n"
            f"- Humidity: {current['relative_humidity_2m']}%\n"
            f"- Wind speed: {current['wind_speed_10m']} km/h\n"
            f"- Condition: {condition}"
        )
    except Exception as e:
        return f"Error getting weather for {city}: {e}"


# ==================================================================
# TOOL 3 — Food & Experience Advisor
# ==================================================================
@tool
def food_experience_advisor(user_request: str, cuisine_preference: str = "", budget: str = "", location: str = "") -> str:
    """Use this when the user asks for restaurant, food, or dining recommendations in Alexandria.
    If cuisine_preference or budget are missing, this returns clarifying questions to ask the
    user instead of a recommendation. CALL THIS TOOL ONLY ONCE per user message, no matter what
    it returns — if it asks for clarification, just relay that question to the user as your
    entire response and stop. Do not call this tool again in the same turn."""
    missing = []
    if not cuisine_preference:
        missing.append("cuisine type (e.g. seafood, Italian, local Egyptian, fast food, fine dining)")
    if not budget:
        missing.append("budget level (cheap, mid-range, or luxury)")

    if missing:
        return "NEED_CLARIFICATION: Ask the user ONE single clear question about: " + " and ".join(missing) + ". Do not call this tool again until they answer."

    query = f"restaurant dining food {cuisine_preference} {budget} {location}".strip()
    retriever = vectordb.as_retriever(search_kwargs={"k": 4})
    docs = retriever.invoke(query)
    context = "\n\n".join([d.page_content for d in docs]) if docs else ""

    if not context.strip():
        return "NO_DATA: Tell the user plainly you don't have restaurant listings for this yet."

    return (
        f"User wants: {cuisine_preference} food, budget {budget}, in {location or 'Alexandria'}.\n\n"
        f"CONTEXT (only use facts that actually appear below — never invent restaurant names "
        f"that aren't in this text):\n\n{context}\n\n"
        f"INSTRUCTIONS: Look for menu items or dishes in the context that ARE genuinely "
        f"'{cuisine_preference}' cuisine. ALWAYS state which restaurant/venue each dish is from "
        f"if that information is present in the context — never list dish names without naming "
        f"the restaurant they belong to. If you find matching items, recommend them directly and "
        f"confidently."
    )


# ==================================================================
# TOOL 4 — Hotel Advisor
# ==================================================================
@tool
def hotel_advisor(user_request: str, budget: str = "", area: str = "") -> str:
    """Use this when the user asks for hotel or accommodation recommendations in Alexandria.
    If budget or preferred area aren't mentioned, you can proceed with general recommendations,
    but mention that asking about budget would help narrow results. Call this ONCE per request."""
    query = f"hotel accommodation {budget} {area} Alexandria".strip()
    retriever = vectordb.as_retriever(search_kwargs={"k": 4})
    docs = retriever.invoke(query)
    context = "\n\n".join([d.page_content for d in docs]) if docs else ""

    if not context.strip():
        return "NO_DATA: Tell the user plainly you don't have hotel listings for this yet."

    return (
        f"CONTEXT (only use hotel names that actually appear below — never invent names not in this text):\n\n{context}\n\n"
        f"Recommend 2-3 hotels grounded ONLY in the above, with a brief reason each."
    )


# ==================================================================
# TOOL 5 — Compare & Decide
# ==================================================================
@tool
def compare_places(place_a: str, place_b: str) -> str:
    """Use this when the user wants to compare two tourist places/attractions in Alexandria to
    decide which to visit (e.g. 'Qaitbay Citadel or Bibliotheca Alexandrina?'). Builds a
    structured Markdown comparison table: History, Time Needed, Price, Best For — followed by
    a one-line recommendation."""
    retriever = vectordb.as_retriever(search_kwargs={"k": 3})
    docs_a = retriever.invoke(place_a)
    docs_b = retriever.invoke(place_b)
    context_a = "\n".join([d.page_content for d in docs_a]) if docs_a else "No specific info found."
    context_b = "\n".join([d.page_content for d in docs_b]) if docs_b else "No specific info found."

    return (
        f"INFO ON '{place_a}':\n{context_a}\n\n"
        f"INFO ON '{place_b}':\n{context_b}\n\n"
        f"Build a Markdown table comparing '{place_a}' vs '{place_b}' with rows: "
        f"History, Time Needed, Price, Best For. If exact details aren't in the info above, "
        f"give a reasonable general estimate and note it's approximate — never invent specific "
        f"facts not supported by the context. End with one sentence recommending which fits "
        f"which type of traveler."
    )


# ==================================================================
# TOOL 6 — Travel Constraints
# ==================================================================
@tool
def apply_travel_constraints(traveler_type: str, original_request: str = "") -> str:
    """Use this when the user mentions their travel situation, such as having kids, traveling
    solo, wanting a cheap/budget trip, traveling as a couple, or elderly travelers. Filters
    knowledge-base content to adapt recommendations accordingly."""
    constraint_hints = {
        "family": "family-friendly, safe, kid-appropriate, not too physically demanding",
        "kids": "family-friendly, safe, kid-appropriate, not too physically demanding",
        "solo": "safe for solo travelers, well-trafficked, easy to navigate alone",
        "budget": "low-cost or free entry, affordable food/transport options",
        "cheap": "low-cost or free entry, affordable food/transport options",
        "couple": "romantic, scenic, good for photos, not overly crowded",
        "elderly": "minimal walking/stairs, accessible, comfortable pace",
    }
    matched_hints = [v for k, v in constraint_hints.items() if k in traveler_type.lower()]
    hint_text = "; ".join(matched_hints) if matched_hints else "general suitability"

    query = f"{original_request} {traveler_type}".strip()
    retriever = vectordb.as_retriever(search_kwargs={"k": 4})
    docs = retriever.invoke(query)
    context = "\n\n".join([d.page_content for d in docs]) if docs else ""

    if not context.strip():
        return "NO_DATA: Tell the user plainly you don't have enough info for this specific request yet."

    return (
        f"TRAVELER TYPE: {traveler_type}\n"
        f"FILTER FOR: {hint_text}\n\n"
        f"RELEVANT CONTEXT:\n{context}\n\n"
        f"Write recommendations adapted for a '{traveler_type}' traveler, applying the filter "
        f"criteria above. Be explicit about why each suggestion fits — never invent facts not "
        f"supported by the context above."
    )


tools = [
    search_tourism_knowledge,
    get_weather,
    food_experience_advisor,
    hotel_advisor,
    compare_places,
    apply_travel_constraints,
]

print(f"✅ {len(tools)} tools ready: {[t.name for t in tools]}")

✅ 6 tools ready: ['search_tourism_knowledge', 'get_weather', 'food_experience_advisor', 'hotel_advisor', 'compare_places', 'apply_travel_constraints']


---
## 6. Build the Orchestrator Agent

In [11]:
AGENT_SYSTEM_PROMPT = """You are TourBot, a friendly and knowledgeable Alexandria (Egypt) tourism assistant.

You have access to these tools:
1. search_tourism_knowledge — places, history, attractions, activities.
2. get_weather — live weather for any Egyptian city.
3. food_experience_advisor — restaurant/dining requests. May ask for cuisine type and budget first.
4. hotel_advisor — hotel/accommodation recommendations. Stay grounded in retrieved data, never invent hotel names.
5. compare_places — compares two named places in a Markdown table.
6. apply_travel_constraints — adapts recommendations for kids, solo, budget, couples, elderly.

How to decide which tool(s) to use:
- Place/activity/history questions → search_tourism_knowledge
- Weather, what to wear, conditions-based suggestions → get_weather
- Food/restaurant/dining requests → food_experience_advisor
- Hotel/accommodation requests → hotel_advisor
- "X or Y?" comparisons between two named places → compare_places
- Mentions of kids, solo, budget, couple, elderly → apply_travel_constraints
- Combine tools when a question needs more than one.

CRITICAL OUTPUT RULES:
- NEVER include "[Source: ...]", "According to the document", URLs, file names, or any citation. State facts naturally as your own knowledge.
- NEVER mention which tool you used or your internal reasoning. Just answer.
- NEVER invent restaurant names, place names, or facts not present in tool results — if data is thin or a cuisine/place isn't covered, say so plainly and offer what IS available instead.
- If food_experience_advisor returns "NEED_CLARIFICATION: ...", ask that question directly and naturally — nothing else.
- If a tool returns "NO_DATA: ...", be honest about the gap, don't fabricate.
- If compare_places returns context, format your reply as a Markdown table (History / Time Needed / Price / Best For) + one recommendation sentence.
- Call each tool only ONCE per user request — never repeat the same tool call.
- Never say things like "it seems the previous response didn't provide enough information" or narrate your own confusion/retry process. If you need to try again internally, just give a clean, confident final answer — the user never sees retries or self-commentary.
- If a tool returns "NEED_CLARIFICATION: ...", ask that ONE question and STOP immediately — never call the same tool again in the same turn, never repeat or rephrase the question multiple times in one response.

TONE RULES — be direct and human, not corporate:
- No filler openers like "I'm sorry, but...", "I hope this helps...", "I wish you...".
- Get straight to the point in the first sentence.
- Talk like a knowledgeable local friend texting back, not a customer service script.
- Respond naturally as if continuing an ongoing chat — don't reintroduce yourself or over-explain.
- Max 3-4 sentences unless the user asked for a list or comparison.
- For greetings, respond briefly without using tools.
- For requests completely unrelated to Alexandria tourism (coding help, math, unrelated general knowledge), politely decline and redirect: say you're focused on helping with Alexandria travel, and ask what they'd like to know about the city. Never actually answer the off-topic request itself.
"""

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", AGENT_SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=False,        # set True only if debugging — keeps the UI clean
    max_iterations=3,
    handle_parsing_errors=True
)

sessions = {}

def ask(question, session_id="default"):
    if session_id not in sessions:
        sessions[session_id] = []
    history = sessions[session_id]

    last_error = None
    for attempt in range(3):
        try:
            q = question if attempt == 0 else question + " (please answer this clearly and directly)"
            result = agent_executor.invoke({"input": q, "chat_history": history})
            answer = result["output"]
            break
        except Exception as e:
            last_error = e
            error_str = str(e)
            if "rate_limit" in error_str.lower() or "429" in error_str:
                answer = "I've hit my usage limit for now — give it a few minutes and try again."
                break
    else:
        print(f"DEBUG: exhausted retries — {repr(last_error)}")
        answer = "Sorry, I had trouble processing that — could you rephrase your question?"

    sessions[session_id].append(HumanMessage(content=question))
    sessions[session_id].append(AIMessage(content=answer))
    return answer

---
## 7. Chat UI

In [12]:
import ipywidgets as widgets
from IPython.display import display, Markdown

text_input = widgets.Text(
    placeholder="Ask me about Alexandria — places, food, weather, comparisons...",
    layout=widgets.Layout(width="70%")
)
send_button  = widgets.Button(description="Send",          button_style="primary")
clear_button = widgets.Button(description="Clear History", button_style="warning")
output       = widgets.Output()

def format_answer(answer):
    if "|" in answer and "---" in answer:   # don't mangle Markdown tables
        return answer.strip()
    answer = re.sub(r'(\d+\.)', r'\n\1', answer)
    answer = re.sub(r'(•|-|\*)\s', r'\n• ', answer)
    answer = re.sub(r'\n{3,}', '\n\n', answer)
    return answer.strip()


def on_send(b):
    question = text_input.value.strip()
    if question:
        text_input.value = ""
        with output:
            print(f"🧑 You: {question}")
            try:
                answer = ask(question, session_id="colab_ui")
                print("🤖 Bot:")
                display(Markdown(answer))  # ← renders tables, bold, lists properly
                print()
            except Exception as e:
                print(f"❌ Error: {e}\n")

def on_clear(b):
    sessions["colab_ui"] = []
    output.clear_output()
    with output:
        print("🗑️ Chat history cleared!\n")

send_button.on_click(on_send)
clear_button.on_click(on_clear)
text_input.on_submit(on_send)

display(widgets.HBox([text_input, send_button, clear_button]), output)

Output()

---
# 9. Test Suite

Run these after Section 6 (the agent must be built first). Each test targets one capability you'll likely be asked about in your graduation defense — tool routing, multi-turn clarification, grounding/anti-hallucination, multi-tool combination, comparison formatting, and error resilience.

Run all test cells, then look over the printed results before your demo/defense.

### Test 1 — Basic RAG retrieval (search_tourism_knowledge)
Confirms the agent correctly routes a plain factual question to the knowledge base and returns a grounded answer.

In [13]:
print("TEST 1 — Basic place/history question\n" + "-"*60)
answer = ask("Tell me about the Citadel of Qaitbay", session_id="test1")
print(textwrap.fill(answer, width=100))

TEST 1 — Basic place/history question
------------------------------------------------------------
The Citadel of Qaitbay, built between 1477‑1479 by Mamluk Sultan Al‑Ashraf Sayf al‑Din Qaitbay, sits
on the exact spot where the ancient Lighthouse of Alexandria once stood. It’s a massive three‑storey
limestone fortress with half‑circular corner towers, an inner courtyard, barracks and a water tank,
designed to guard the Eastern Harbour against naval attacks. Today the citadel is intact, houses a
small maritime museum, and offers panoramic Mediterranean views—perfect for a quick history stop and
great photo ops.


### Test 2 — Live tool integration (get_weather)
Confirms the agent calls an external real-time API correctly, not just the static knowledge base.

In [14]:
print("TEST 2 — Live weather API call\n" + "-"*60)
answer = ask("What's the weather like in Alexandria right now?", session_id="test2")
print(textwrap.fill(answer, width=100))

TEST 2 — Live weather API call
------------------------------------------------------------
Currently it's around 23 °C, clear skies, 65 % humidity and a gentle 6 km/h breeze. Perfect for a
stroll along the Corniche.


### Test 3 — Multi-turn clarification flow (food_experience_advisor)
Confirms the agent asks for missing slot-filling info (cuisine + budget) instead of guessing, then gives a grounded recommendation once both are supplied.

In [15]:
print("TEST 3a — Should ask a clarifying question (no cuisine/budget given)\n" + "-"*60)
answer = ask("Where should I eat in Alexandria?", session_id="test3")
print(textwrap.fill(answer, width=100))
assert "?" in answer, "⚠️ Expected a clarifying question, got a direct answer instead."
print("\n✅ Correctly asked for missing info.\n")

print("TEST 3b — Follow-up with cuisine + budget supplied\n" + "-"*60)
answer = ask("Seafood, mid-range budget", session_id="test3")
print(textwrap.fill(answer, width=100))

TEST 3a — Should ask a clarifying question (no cuisine/budget given)
------------------------------------------------------------
What type of cuisine are you craving and what budget range are you looking for (cheap, mid‑range, or
luxury)?

✅ Correctly asked for missing info.

TEST 3b — Follow-up with cuisine + budget supplied
------------------------------------------------------------
Try the seasonal beachfront restaurant – it’s mid‑range and perfect for a relaxed seafood meal by
the sea. Their grilled seafood platter (shrimp, calamari, fish fillets, crab, mussels) with creamy
dill‑lemon sauce is a standout, and the fresh seafood‑mixed basmati rice (jumbo shrimp, veggies,
fritters) is both hearty and flavorful. If you’re after something classic, the grilled salmon fillet
(≈ 980 EGP) or the fish‑of‑the‑day with lemon‑dill sauce (≈ 970 EGP) are excellent choices.


### Test 4 — Anti-hallucination / honesty check
Confirms the bot does NOT invent a restaurant/place name when the requested cuisine isn't covered by the knowledge base — it should say so plainly instead of fabricating.

In [16]:
print("TEST 4 — Cuisine likely not in knowledge base (checks for honest gap-flagging)\n" + "-"*60)
answer = ask("I want Japanese sushi, budget around 2000 EGP", session_id="test4")
print(textwrap.fill(answer, width=100))
print("\n👉 Manually verify: does the bot admit it doesn't have sushi-specific listings,")
print("   rather than inventing a restaurant name? This is the key anti-hallucination check.")

TEST 4 — Cuisine likely not in knowledge base (checks for honest gap-flagging)
------------------------------------------------------------
The menu you shared lists several sushi options—8‑piece rolls for 415 EGP, a 385 EGP shrimp roll,
and bowls around 340‑375 EGP—but it doesn’t mention the restaurant name. Since I can’t name a
specific venue, I recommend looking for sushi spots in Alexandria that offer similar‑priced rolls
(around 400‑450 EGP) and bowls (≈ 350 EGP), which will comfortably fit your 2,000 EGP budget. If you
need help locating particular sushi restaurants, just let me know!

👉 Manually verify: does the bot admit it doesn't have sushi-specific listings,
   rather than inventing a restaurant name? This is the key anti-hallucination check.


### Test 5 — Source/citation leakage check
Confirms the bot never exposes internal `[Source: ...]` tags, URLs, or "according to the document" phrasing — a common RAG implementation flaw.

In [17]:
print("TEST 5 — Clean output check\n" + "-"*60)
answer = ask("What is the history of the Bibliotheca Alexandrina?", session_id="test5")
print(textwrap.fill(answer, width=100))

leak_markers = ["[Source:", "according to the document", "http://", "https://", ".pdf", ".html"]
leaked = [m for m in leak_markers if m.lower() in answer.lower()]
assert not leaked, f"⚠️ Leakage detected: {leaked}"
print("\n✅ No source/citation leakage detected.")

TEST 5 — Clean output check
------------------------------------------------------------
The Bibliotheca Alexandrina was conceived after the 1990 Vienna Declaration, which called for a
modern “new library of Alexandria,” and was reinforced by the Aswan Declaration the same year.
Egyptian scholar Prof. Mostafa El‑Abbadi championed the idea, and UNESCO, along with the Egyptian
government, launched the project. Construction began in 1995, and the futuristic disc‑shaped
building opened its doors in 2002, marking the revival of the ancient Library of Alexandria’s
legacy. Over the past two decades it’s become a cultural hub for research, exhibitions, and
international dialogue, celebrating its 20th anniversary in 2022.

✅ No source/citation leakage detected.


### Test 6 — Structured comparison output (compare_places)
Confirms the agent produces a well-formed Markdown table when asked to compare two named places — important since your frontend renders this as a real table.

In [18]:
print("TEST 6 — Compare two places (expect Markdown table)\n" + "-"*60)
answer = ask("Qaitbay Citadel or Montaza Palace — which should I visit?", session_id="test6")
print(answer)

assert "|" in answer and "---" in answer, "⚠️ Expected a Markdown table, none detected."
print("\n✅ Markdown table structure detected.")

TEST 6 — Compare two places (expect Markdown table)
------------------------------------------------------------
**Qaitbay Citadel vs Montaza Palace**

|                     | **Qaitbay Citadel** | **Montaza Palace** |
|---------------------|---------------------|--------------------|
| **History** | 15th‑century Mamluk fortress built 1477‑1479 on the site of the ancient Lighthouse of Alexandria. | Late 19th‑century royal summer residence (Salamlek Palace 1892, expanded 1932) of the Muhammad Ali dynasty, now a public museum and park. |
| **Time Needed** | 1–2 hours to explore the fort, museum, and sea views. | 2–3 hours to wander the palace interiors, gardens, and beach area. |
| **Price** | Modest entry fee (around EGP 30–50). | Small admission (≈ EGP 20–40) for the museum; the gardens are free. |
| **Best For** | History buffs and those wanting iconic coastal views. | Families, picnickers, and anyone who enjoys gardens, beach, and relaxed strolling. |

**Recommendation:** If you’re k

### Test 7 — Traveler-constraint adaptation (apply_travel_constraints)
Confirms recommendations actually change based on stated traveler type (family with kids vs. solo budget traveler) rather than giving the same generic answer.

In [19]:
print("TEST 7a — Family with kids\n" + "-"*60)
answer_family = ask("I'm traveling with my kids, what should we visit in Alexandria?", session_id="test7a")
print(textwrap.fill(answer_family, width=100))

print("\n\nTEST 7b — Solo budget traveler\n" + "-"*60)
answer_solo = ask("I'm a solo traveler on a tight budget, what should I visit in Alexandria?", session_id="test7b")
print(textwrap.fill(answer_solo, width=100))

print("\n👉 Manually verify: the two answers should differ meaningfully (family-safe vs budget-friendly framing),")
print("   not be the same generic list with different wording.")

TEST 7a — Family with kids
------------------------------------------------------------
Check out the Alexandria Aquarium Museum – kids love the colorful marine life and interactive
displays, and it’s an easy, safe indoor spot.   Stroll the Corniche and stop at the historic
Qaitbay Citadel; the fortress feels like a real‑life castle and the open‑air views keep little ones
entertained without a long walk.   Visit the Bibliotheca Alexandrina’s Children’s Corner, where
story‑time sessions and hands‑on exhibits make learning about the city’s history fun for younger
visitors.


TEST 7b — Solo budget traveler
------------------------------------------------------------
Check out the free‑entry spots that let you soak up Alexandria’s history without spending much:
stroll the Corniche for sea views and street‑life, wander the historic downtown square, and visit
the Roman Amphitheater (outside the ticketed area). The Bibliotheca Alexandrina’s main lobby and its
public exhibitions are low‑cost, 

### Test 8 — Multi-tool combination in a single turn
Confirms the orchestrator can decide to call MORE THAN ONE tool for a single message when the question genuinely requires it (weather + place recommendation together) — the core "agentic" behavior, not just simple Q&A.

In [20]:
print("TEST 8 — Combined weather + recommendation (multi-tool reasoning)\n" + "-"*60)
answer = ask("What's the weather like today, and what should I do in Alexandria given that?", session_id="test8")
print(textwrap.fill(answer, width=100))
print("\n👉 Manually verify: does the answer reference both the actual weather conditions")
print("   AND a place/activity recommendation that makes sense given those conditions?")

TEST 8 — Combined weather + recommendation (multi-tool reasoning)
------------------------------------------------------------
It’s around 23 °C with clear skies and a gentle breeze—perfect for strolling outdoors. Head to the
Corniche for a seaside walk, then explore Qaitbay Citadel or the Bibliotheca Alexandrina for a mix
of history and culture. If you’re hungry, drop by a waterfront café for fresh seafood while you
enjoy the view.

👉 Manually verify: does the answer reference both the actual weather conditions
   AND a place/activity recommendation that makes sense given those conditions?


### Test 9 — Off-topic / out-of-scope handling
Confirms the bot stays in its tourism-assistant role for unrelated questions instead of trying to force an irrelevant tool call, and that greetings don't trigger unnecessary tool use.

In [21]:
print("TEST 9a — Plain greeting (should NOT trigger any tool call)\n" + "-"*60)
answer = ask("hi", session_id="test9a")
print(textwrap.fill(answer, width=100))

print("\n\nTEST 9b — Off-topic question (should decline gracefully, stay in character)\n" + "-"*60)
answer = ask("Can you help me write a Python script to sort a list?", session_id="test9b")
print(textwrap.fill(answer, width=100))

TEST 9a — Plain greeting (should NOT trigger any tool call)
------------------------------------------------------------
Hey! What would you like to know about Alexandria?


TEST 9b — Off-topic question (should decline gracefully, stay in character)
------------------------------------------------------------
I’m focused on helping with Alexandria travel. What would you like to know about the city?


### Test 10 — Multi-turn memory / context retention
Confirms the agent correctly uses `chat_history` to resolve references across turns (e.g. "there", "it") rather than treating every message as stateless.

In [22]:
print("TEST 10 — Context retention across turns\n" + "-"*60)
a1 = ask("Tell me about Montaza Palace", session_id="test10")
print("Turn 1:", textwrap.fill(a1, width=100))

a2 = ask("How much time do I need there?", session_id="test10")
print("\nTurn 2:", textwrap.fill(a2, width=100))
print("\n👉 Manually verify: Turn 2 should correctly understand 'there' = Montaza Palace,")
print("   not ask 'where?' or answer about a different/generic place.")

TEST 10 — Context retention across turns
------------------------------------------------------------
Turn 1: Montaza Palace, perched on a low plateau in Alexandria’s Montaza district, was built in 1892 by
Khedive Abbas II as a hunting lodge and summer residence. Designed in an Italian‑Renaissance style
by architect Hassan Pasha El‑Adawi, the palace sits amid the expansive Montaza Gardens overlooking
the Mediterranean coast. Today it serves as a museum and occasional official venue, while the
surrounding gardens are open to the public for leisurely walks, picnics, and seaside views. It’s a
must‑see spot for anyone interested in royal history and a tranquil escape from the city bustle.

Turn 2: Plan on spending 2–3 hours to see the palace rooms and wander the Montaza Gardens; if you want to
add a beach stop or a relaxed picnic, give yourself about 4 hours for a comfortable half‑day visit.

👉 Manually verify: Turn 2 should correctly understand 'there' = Montaza Palace,
   not ask 'where?

### Test 11 — Error resilience
Confirms the system degrades gracefully (no raw stack traces shown to the user) even with a malformed/edge-case input.

In [23]:
print("TEST 11 — Edge case / empty-ish input handling\n" + "-"*60)
edge_cases = ["???", "asdkjasldkj", ""]
for case in edge_cases:
    if not case.strip():
        print(f"  (skipped empty string — UI layer already blocks this before it reaches ask())")
        continue
    try:
        answer = ask(case, session_id="test11")
        print(f"  Input: {case!r} → Bot: {answer[:150]}")
    except Exception as e:
        print(f"  ❌ Input: {case!r} raised an unhandled exception: {e}")
print("\n✅ No unhandled exceptions should appear above.")

TEST 11 — Edge case / empty-ish input handling
------------------------------------------------------------
  Input: '???' → Bot: What can I help you with about Alexandria?
  Input: 'asdkjasldkj' → Bot: Let me know what you’d like to discover or plan in Alexandria!
  (skipped empty string — UI layer already blocks this before it reaches ask())

✅ No unhandled exceptions should appear above.
